In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
import pandas as pd

DATA_PATH = "../data/processed/technical_feature_matrix.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(9776, 23)

In [3]:
X = df.drop(columns=["label"])
y = df["label"]

print("X:", X.shape)
print("y:", y.shape)

X: (9776, 22)
y: (9776,)


In [4]:
df

,semantic_similarity,tfidf_similarity,word_overlap,resume_skill_count,job_skill_count,matched_skill_count,skill_match_ratio,skill_coverage,resume_education_level,job_education_level,...,experience_match,resume_core_cs_count,job_core_cs_count,resume_degree_count,job_degree_count,resume_length,job_description_length,resume_word_count,job_word_count,label
0,0.507104,0.056482,0.133333,4,0,0,0.000000,0.000000,5,0,...,1.0,1,0,1,0,286,696,37,99,1
1,0.562818,0.299446,0.750000,14,4,4,1.000000,0.285714,3,0,...,1.0,4,2,1,0,2506,107,333,16,1
2,0.523189,0.054792,0.133333,5,0,0,0.000000,0.000000,2,0,...,1.0,1,0,0,0,287,696,37,99,1
3,0.529498,0.234125,0.533333,7,0,0,0.000000,0.000000,4,0,...,1.0,1,0,2,0,3143,91,401,16,1
4,0.471209,0.047656,0.117647,1,0,0,0.000000,0.000000,0,3,...,1.0,0,0,0,1,277,616,37,90,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9771,0.537629,0.358228,0.476190,9,0,0,0.000000,0.000000,3,0,...,1.0,1,0,1,0,2655,157,343,24,0
9772,0.711359,0.186097,0.533333,3,0,0,0.000000,0.000000,3,0,...,1.0,0,0,1,0,3242,105,428,15,1
9773,0.585595,0.203442,0.315789,2,0,0,0.000000,0.000000,0,0,...,1.0,0,0,0,0,466,125,52,18,1
9774,0.588449,0.085541,0.223881,1,3,1,0.333333,1.000000,5,0,...,1.0,0,1,1,0,306,664,38,90,0


### Train/Test Splitting

In [5]:
NORMALIZED_PATH = "../data/processed/technical_normalized_dataset.csv"

normalized_df = pd.read_csv(NORMALIZED_PATH)

print("Normalized dataset shape:", normalized_df.shape)

Normalized dataset shape: (9776, 11)


In [6]:
assert len(normalized_df) == len(df)

print("✓ Row counts match")

✓ Row counts match


In [7]:
assert normalized_df["label"].equals(df["label"])

print("✓ Labels match")

✓ Labels match


In [8]:
groups = normalized_df["job_description"].fillna("").astype(str)

print("Total rows:", len(groups))
print("Unique job descriptions:", groups.nunique())

Total rows: 9776
Unique job descriptions: 2525


In [9]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 7875
Testing samples : 1901


For no JD leakage

In [10]:
train_jobs = set(
    groups.iloc[train_idx]
)

test_jobs = set(
    groups.iloc[test_idx]
)

overlap = train_jobs.intersection(test_jobs)

print("Train job descriptions:", len(train_jobs))
print("Test job descriptions :", len(test_jobs))
print("Overlapping job descriptions:", len(overlap))

assert len(overlap) == 0

print("✓ No job-description leakage")

Train job descriptions: 2020
Test job descriptions : 505
Overlapping job descriptions: 0
✓ No job-description leakage


In [11]:
print("Training label distribution:")
print(y_train.value_counts())
print()

print("Training label percentages:")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print()

print("Test label distribution:")
print(y_test.value_counts())
print()

print("Test label percentages:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training label distribution:
label
0    4095
1    3780
Name: count, dtype: int64

Training label percentages:
label
0    52.0
1    48.0
Name: proportion, dtype: float64

Test label distribution:
label
1    978
0    923
Name: count, dtype: int64

Test label percentages:
label
1    51.45
0    48.55
Name: proportion, dtype: float64


### Logistic Regression

In [12]:
logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Training Logistic Regression...")

logistic_model.fit(
    X_train,
    y_train
)

print("✓ Training complete")

Training Logistic Regression...
✓ Training complete


In [13]:
#Predictions

y_pred = logistic_model.predict(X_test)

y_prob = logistic_model.predict_proba(X_test)[:, 1]

In [14]:
#Evaluate

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

print("=" * 60)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

LOGISTIC REGRESSION RESULTS
Accuracy : 0.5034
Precision: 0.5209
Recall   : 0.4335
F1 Score : 0.4732
ROC-AUC  : 0.5086


In [15]:
#Classification report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Rejected",
            "Accepted"
        ]
    )
)

cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

              precision    recall  f1-score   support

    Rejected       0.49      0.58      0.53       923
    Accepted       0.52      0.43      0.47       978

    accuracy                           0.50      1901
   macro avg       0.51      0.51      0.50      1901
weighted avg       0.51      0.50      0.50      1901

Confusion Matrix:
[[533 390]
 [554 424]]


Findings : Yeah this model is basically random and theres no point in this

Our current 22-feature representation isn't giving Logistic Regression useful linear separation between accepted/rejected candidates.

#### Feature Groups

In [16]:
SEMANTIC_FEATURES = [
    "semantic_similarity"
]

LEXICAL_FEATURES = [
    "tfidf_similarity",
    "word_overlap"
]

EXPLICIT_FEATURES = [
    "resume_skill_count",
    "job_skill_count",
    "matched_skill_count",
    "skill_match_ratio",
    "skill_coverage",
    "resume_education_level",
    "job_education_level",
    "education_match",
    "resume_experience_years",
    "job_required_experience_years",
    "experience_match",
    "resume_core_cs_count",
    "job_core_cs_count",
    "resume_degree_count",
    "job_degree_count",
    "resume_length",
    "job_description_length",
    "resume_word_count",
    "job_word_count"
]

In [17]:
print("Semantic:", len(SEMANTIC_FEATURES))
print("Lexical :", len(LEXICAL_FEATURES))
print("Explicit:", len(EXPLICIT_FEATURES))

assert len(
    SEMANTIC_FEATURES
    + LEXICAL_FEATURES
    + EXPLICIT_FEATURES
) == 22

print("✓ Feature groups contain all 22 features")

Semantic: 1
Lexical : 2
Explicit: 19
✓ Feature groups contain all 22 features


In [18]:
def evaluate_logistic_regression(
    feature_columns,
    experiment_name
):
    X_train_exp = X_train[feature_columns]
    X_test_exp = X_test[feature_columns]

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    model.fit(
        X_train_exp,
        y_train
    )

    predictions = model.predict(
        X_test_exp
    )

    probabilities = model.predict_proba(
        X_test_exp
    )[:, 1]

    results = {
        "experiment": experiment_name,
        "features": len(feature_columns),
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "precision": precision_score(
            y_test,
            predictions
        ),
        "recall": recall_score(
            y_test,
            predictions
        ),
        "f1": f1_score(
            y_test,
            predictions
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        )
    }

    return results

In [19]:
experiments = [

    (
        "Semantic",
        SEMANTIC_FEATURES
    ),

    (
        "Lexical",
        LEXICAL_FEATURES
    ),

    (
        "Explicit",
        EXPLICIT_FEATURES
    ),

    (
        "Semantic + Lexical",
        SEMANTIC_FEATURES + LEXICAL_FEATURES
    ),

    (
        "Semantic + Explicit",
        SEMANTIC_FEATURES + EXPLICIT_FEATURES
    ),

    (
        "Lexical + Explicit",
        LEXICAL_FEATURES + EXPLICIT_FEATURES
    ),

    (
        "All Features",
        SEMANTIC_FEATURES
        + LEXICAL_FEATURES
        + EXPLICIT_FEATURES
    )
]


results = []

for name, features in experiments:

    print(
        f"Running: {name}"
    )

    result = evaluate_logistic_regression(
        features,
        name
    )

    results.append(result)

Running: Semantic
Running: Lexical
Running: Explicit
Running: Semantic + Lexical
Running: Semantic + Explicit
Running: Lexical + Explicit
Running: All Features


c:\Users\BHAVANI\anaconda3\envs\ai-hire\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [20]:
results_df = pd.DataFrame(results)

results_df

,experiment,features,accuracy,precision,recall,f1,roc_auc
0,Semantic,1,0.485534,0.000000,0.000000,0.000000,0.527197
1,Lexical,2,0.493951,0.518692,0.226994,0.315789,0.496522
2,Explicit,19,0.499737,0.518934,0.378323,0.437611,0.499327
3,Semantic + Lexical,3,0.501315,0.541436,0.200409,0.292537,0.521373
4,Semantic + Explicit,20,0.501315,0.521490,0.372188,0.434368,0.506242
5,Lexical + Explicit,21,0.500789,0.517835,0.430470,0.470128,0.502631
6,All Features,22,0.503419,0.520885,0.433538,0.473214,0.508644


In [21]:
results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

,experiment,features,accuracy,precision,recall,f1,roc_auc
0,Semantic,1,0.485534,0.000000,0.000000,0.000000,0.527197
1,Semantic + Lexical,3,0.501315,0.541436,0.200409,0.292537,0.521373
2,All Features,22,0.503419,0.520885,0.433538,0.473214,0.508644
3,Semantic + Explicit,20,0.501315,0.521490,0.372188,0.434368,0.506242
4,Lexical + Explicit,21,0.500789,0.517835,0.430470,0.470128,0.502631
5,Explicit,19,0.499737,0.518934,0.378323,0.437611,0.499327
6,Lexical,2,0.493951,0.518692,0.226994,0.315789,0.496522


Yeahh no this model is not the best for this

Or we have to clean our dataset a bit

### Random Forest

In [22]:
from sklearn.ensemble import RandomForestClassifier

In [23]:
def evaluate_random_forest(
    feature_columns,
    experiment_name
):
    X_train_exp = X_train[feature_columns]
    X_test_exp = X_test[feature_columns]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    model.fit(
        X_train_exp,
        y_train
    )

    predictions = model.predict(
        X_test_exp
    )

    probabilities = model.predict_proba(
        X_test_exp
    )[:, 1]

    return {
        "experiment": experiment_name,
        "features": len(feature_columns),
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        )
    }

In [24]:
rf_results = []

for name, features in experiments:

    print(
        f"Running Random Forest: {name}"
    )

    result = evaluate_random_forest(
        features,
        name
    )

    rf_results.append(result)

rf_results_df = pd.DataFrame(
    rf_results
)

rf_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

Running Random Forest: Semantic
Running Random Forest: Lexical
Running Random Forest: Explicit
Running Random Forest: Semantic + Lexical
Running Random Forest: Semantic + Explicit
Running Random Forest: Lexical + Explicit
Running Random Forest: All Features


,experiment,features,accuracy,precision,recall,f1,roc_auc
0,Semantic + Explicit,20,0.571278,0.596679,0.514315,0.552444,0.615031
1,All Features,22,0.560231,0.582944,0.510225,0.544166,0.604328
2,Explicit,19,0.559179,0.588384,0.476483,0.526554,0.601868
3,Lexical + Explicit,21,0.562862,0.584971,0.517382,0.549105,0.597796
4,Semantic + Lexical,3,0.525513,0.538540,0.542945,0.540733,0.559710
5,Lexical,2,0.522883,0.537018,0.526585,0.531750,0.533168
6,Semantic,1,0.510258,0.525627,0.492843,0.508707,0.518223


### XGBoost

In [25]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import pandas as pd



In [26]:
def run_xgboost_experiment(
    feature_names,
    feature_set_name
):

    X_train_subset = X_train[
        feature_names
    ]

    X_test_subset = X_test[
        feature_names
    ]

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss"
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model.fit(
        X_train_subset,
        y_train
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test_subset
    )

    y_prob = model.predict_proba(
        X_test_subset
    )[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "features": feature_set_name,
        "feature_count": len(feature_names),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "model": model
    }

In [27]:
xgb_results = []

for name, features in experiments:
    print(
            f"Running Random Forest: {name}"
        )
    result = run_xgboost_experiment(
        features,
        name
    )

    xgb_results.append(
        result
    )

Running Random Forest: Semantic
Running Random Forest: Lexical
Running Random Forest: Explicit
Running Random Forest: Semantic + Lexical
Running Random Forest: Semantic + Explicit
Running Random Forest: Lexical + Explicit
Running Random Forest: All Features


In [28]:

xgb_results_df = pd.DataFrame(
    xgb_results
)

xgb_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

,features,feature_count,accuracy,precision,recall,f1,roc_auc,model
0,Semantic + Explicit,20,0.569174,0.603922,0.472393,0.530120,0.624832,"XGBClassifier(base_score=None, booster=None, c..."
1,All Features,22,0.568648,0.590805,0.525562,0.556277,0.616808,"XGBClassifier(base_score=None, booster=None, c..."
2,Explicit,19,0.571804,0.596698,0.517382,0.554217,0.616334,"XGBClassifier(base_score=None, booster=None, c..."
3,Lexical + Explicit,21,0.561284,0.588670,0.488753,0.534078,0.607494,"XGBClassifier(base_score=None, booster=None, c..."
4,Semantic + Lexical,3,0.529195,0.560408,0.393661,0.462462,0.565118,"XGBClassifier(base_score=None, booster=None, c..."
5,Lexical,2,0.533403,0.559790,0.435583,0.489937,0.550103,"XGBClassifier(base_score=None, booster=None, c..."
6,Semantic,1,0.512362,0.548757,0.293456,0.382412,0.535348,"XGBClassifier(base_score=None, booster=None, c..."


### Gradient Boost

In [29]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
import pandas as pd


In [30]:
def run_gradient_boosting(
    X_train,
    X_test,
    y_train,
    y_test,
    feature_columns,
    feature_set_name
):
    """
    Train and evaluate a Gradient Boosting classifier
    using a specific feature set.
    """

    # --------------------------------------------------------
    # Select features
    # --------------------------------------------------------

    X_train_selected = X_train[
        feature_columns
    ]

    X_test_selected = X_test[
        feature_columns
    ]

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model.fit(
        X_train_selected,
        y_train
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test_selected
    )

    y_prob = model.predict_proba(
        X_test_selected
    )[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "features": feature_set_name,
        "feature_count": len(feature_columns),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "model": model,
    }

In [31]:
gb_results = []

for name, features in experiments:
    print(
            f"Running Gradient Boost: {name}"
        )
    result = run_gradient_boosting(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            feature_columns=features,
            feature_set_name=name
        )

    gb_results.append(result)

Running Gradient Boost: Semantic
Running Gradient Boost: Lexical
Running Gradient Boost: Explicit
Running Gradient Boost: Semantic + Lexical
Running Gradient Boost: Semantic + Explicit
Running Gradient Boost: Lexical + Explicit
Running Gradient Boost: All Features


In [32]:
gb_results_df = pd.DataFrame(
    gb_results
)

gb_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

,features,feature_count,accuracy,precision,recall,f1,roc_auc,model
0,Semantic + Explicit,20,0.568648,0.599747,0.485685,0.536723,0.629280,([DecisionTreeRegressor(criterion='friedman_ms...
1,All Features,22,0.574435,0.600476,0.516360,0.555250,0.625934,([DecisionTreeRegressor(criterion='friedman_ms...
2,Explicit,19,0.558653,0.585276,0.487730,0.532069,0.612361,([DecisionTreeRegressor(criterion='friedman_ms...
3,Lexical + Explicit,21,0.564966,0.584736,0.532720,0.557517,0.610117,([DecisionTreeRegressor(criterion='friedman_ms...
4,Semantic + Lexical,3,0.540242,0.579511,0.387526,0.464461,0.571588,([DecisionTreeRegressor(criterion='friedman_ms...
5,Lexical,2,0.534982,0.558750,0.457055,0.502812,0.550452,([DecisionTreeRegressor(criterion='friedman_ms...
6,Semantic,1,0.528669,0.579767,0.304703,0.399464,0.544973,([DecisionTreeRegressor(criterion='friedman_ms...


### Tuned features

In [33]:
# ============================================================
# MODEL TUNING IMPORTS
# ============================================================

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier

import pandas as pd

In [34]:
# ============================================================
# CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

#We're using StratifiedKFold because this is binary classification and we want each fold to maintain 
#roughly the same accepted/rejected ratio.


In [35]:
# ============================================================
# XGBOOST HYPERPARAMETER TUNING
# ============================================================

def tune_xgboost(
    X_train,
    y_train,
    feature_columns
):
    """
    Tune XGBoost using 5-fold stratified CV.
    """

    X_train_selected = X_train[
        feature_columns
    ]

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    param_distributions = {

        "n_estimators": [
            100,
            200,
            300,
            500
        ],

        "learning_rate": [
            0.01,
            0.03,
            0.05,
            0.1
        ],

        "max_depth": [
            2,
            3,
            4,
            5,
            6
        ],

        "min_child_weight": [
            1,
            3,
            5,
            10
        ],

        "subsample": [
            0.7,
            0.8,
            0.9,
            1.0
        ],

        "colsample_bytree": [
            0.7,
            0.8,
            0.9,
            1.0
        ],

        "gamma": [
            0,
            0.1,
            0.3,
            0.5
        ],

        "reg_alpha": [
            0,
            0.01,
            0.1,
            1
        ],

        "reg_lambda": [
            1,
            2,
            5,
            10
        ]
    }

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=30,
        scoring="roc_auc",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(
        X_train_selected,
        y_train
    )

    print()
    print("=" * 70)
    print("XGBOOST TUNING RESULTS")
    print("=" * 70)

    print(
        f"Best CV ROC-AUC: "
        f"{search.best_score_:.6f}"
    )

    print()
    print("Best parameters:")

    for parameter, value in search.best_params_.items():

        print(
            f"  {parameter}: {value}"
        )

    return search

In [36]:
# ============================================================
# GRADIENT BOOSTING HYPERPARAMETER TUNING
# ============================================================

def tune_gradient_boosting(
    X_train,
    y_train,
    feature_columns
):
    """
    Tune Gradient Boosting using 5-fold stratified CV.
    """

    X_train_selected = X_train[
        feature_columns
    ]

    model = GradientBoostingClassifier(
        random_state=42
    )

    param_distributions = {

        "n_estimators": [
            50,
            100,
            200,
            300,
            500
        ],

        "learning_rate": [
            0.01,
            0.03,
            0.05,
            0.1,
            0.2
        ],

        "max_depth": [
            1,
            2,
            3,
            4,
            5
        ],

        "min_samples_split": [
            2,
            5,
            10,
            20
        ],

        "min_samples_leaf": [
            1,
            2,
            5,
            10
        ],

        "subsample": [
            0.7,
            0.8,
            0.9,
            1.0
        ],

        "max_features": [
            None,
            "sqrt",
            "log2"
        ]
    }

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=30,
        scoring="roc_auc",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(
        X_train_selected,
        y_train
    )

    print()
    print("=" * 70)
    print("GRADIENT BOOSTING TUNING RESULTS")
    print("=" * 70)

    print(
        f"Best CV ROC-AUC: "
        f"{search.best_score_:.6f}"
    )

    print()
    print("Best parameters:")

    for parameter, value in search.best_params_.items():

        print(
            f"  {parameter}: {value}"
        )

    return search

In [37]:
TUNING_FEATURE_GROUPS = {

    "Semantic + Explicit": (
        SEMANTIC_FEATURES
        + EXPLICIT_FEATURES
    ),

    "All Features": (
        SEMANTIC_FEATURES
        + LEXICAL_FEATURES
        + EXPLICIT_FEATURES
    )
}

In [38]:
# ============================================================
# TUNE XGBOOST
# ============================================================

xgb_tuning_results = {}

for feature_set_name, feature_columns in TUNING_FEATURE_GROUPS.items():

    print()
    print("=" * 70)
    print(
        f"TUNING XGBOOST: {feature_set_name}"
    )
    print("=" * 70)

    search = tune_xgboost(
        X_train=X_train,
        y_train=y_train,
        feature_columns=feature_columns
    )

    xgb_tuning_results[
        feature_set_name
    ] = search


TUNING XGBOOST: Semantic + Explicit
Fitting 5 folds for each of 30 candidates, totalling 150 fits

XGBOOST TUNING RESULTS
Best CV ROC-AUC: 0.595194

Best parameters:
  subsample: 1.0
  reg_lambda: 2
  reg_alpha: 0.01
  n_estimators: 200
  min_child_weight: 3
  max_depth: 6
  learning_rate: 0.05
  gamma: 0.5
  colsample_bytree: 1.0

TUNING XGBOOST: All Features
Fitting 5 folds for each of 30 candidates, totalling 150 fits

XGBOOST TUNING RESULTS
Best CV ROC-AUC: 0.599576

Best parameters:
  subsample: 0.8
  reg_lambda: 5
  reg_alpha: 1
  n_estimators: 200
  min_child_weight: 1
  max_depth: 6
  learning_rate: 0.05
  gamma: 0.3
  colsample_bytree: 0.8


In [39]:
# ============================================================
# TUNE GRADIENT BOOSTING
# ============================================================

gb_tuning_results = {}

for feature_set_name, feature_columns in TUNING_FEATURE_GROUPS.items():

    print()
    print("=" * 70)
    print(
        f"TUNING GRADIENT BOOSTING: "
        f"{feature_set_name}"
    )
    print("=" * 70)

    search = tune_gradient_boosting(
        X_train=X_train,
        y_train=y_train,
        feature_columns=feature_columns
    )

    gb_tuning_results[
        feature_set_name
    ] = search


TUNING GRADIENT BOOSTING: Semantic + Explicit
Fitting 5 folds for each of 30 candidates, totalling 150 fits

GRADIENT BOOSTING TUNING RESULTS
Best CV ROC-AUC: 0.592033

Best parameters:
  subsample: 0.8
  n_estimators: 300
  min_samples_split: 10
  min_samples_leaf: 10
  max_features: None
  max_depth: 5
  learning_rate: 0.2

TUNING GRADIENT BOOSTING: All Features
Fitting 5 folds for each of 30 candidates, totalling 150 fits

GRADIENT BOOSTING TUNING RESULTS
Best CV ROC-AUC: 0.593811

Best parameters:
  subsample: 0.7
  n_estimators: 200
  min_samples_split: 10
  min_samples_leaf: 10
  max_features: None
  max_depth: 4
  learning_rate: 0.1


In [40]:
# ============================================================
# EVALUATE TUNED MODELS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


def evaluate_tuned_model(
    search,
    X_test,
    y_test,
    feature_columns,
    model_name,
    feature_set_name
):

    X_test_selected = X_test[
        feature_columns
    ]

    model = search.best_estimator_

    y_pred = model.predict(
        X_test_selected
    )

    y_prob = model.predict_proba(
        X_test_selected
    )[:, 1]

    return {
        "model": model_name,
        "features": feature_set_name,
        "feature_count": len(feature_columns),

        "accuracy": accuracy_score(
            y_test,
            y_pred
        ),

        "precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y_test,
            y_prob
        )
    }

In [41]:
# ============================================================
# FINAL TUNED MODEL EVALUATION
# ============================================================

tuned_results = []

for feature_set_name, feature_columns in TUNING_FEATURE_GROUPS.items():

    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    result = evaluate_tuned_model(
        search=xgb_tuning_results[
            feature_set_name
        ],

        X_test=X_test,
        y_test=y_test,

        feature_columns=feature_columns,

        model_name="XGBoost",

        feature_set_name=feature_set_name
    )

    tuned_results.append(result)

    # --------------------------------------------------------
    # Gradient Boosting
    # --------------------------------------------------------

    result = evaluate_tuned_model(
        search=gb_tuning_results[
            feature_set_name
        ],

        X_test=X_test,
        y_test=y_test,

        feature_columns=feature_columns,

        model_name="Gradient Boosting",

        feature_set_name=feature_set_name
    )

    tuned_results.append(result)


tuned_results_df = pd.DataFrame(
    tuned_results
)

tuned_results_df = (
    tuned_results_df
    .sort_values(
        "roc_auc",
        ascending=False
    )
    .reset_index(drop=True)
)


print()
print("=" * 70)
print("TUNED MODEL COMPARISON")
print("=" * 70)

print(
    tuned_results_df.to_string(
        index=False
    )
)


TUNED MODEL COMPARISON
            model            features  feature_count  accuracy  precision   recall       f1  roc_auc
          XGBoost Semantic + Explicit             20  0.567596   0.593750 0.505112 0.545856 0.621271
          XGBoost        All Features             22  0.563914   0.583991 0.529652 0.555496 0.615040
Gradient Boosting        All Features             22  0.545502   0.567376 0.490798 0.526316 0.603348
Gradient Boosting Semantic + Explicit             20  0.551289   0.572254 0.506135 0.537168 0.602780


In [42]:
for feature_set_name, search in xgb_tuning_results.items():

    model = search.best_estimator_

    feature_columns = TUNING_FEATURE_GROUPS[
        feature_set_name
    ]

    importance = pd.DataFrame({
        "feature": feature_columns,
        "importance": model.feature_importances_
    }).sort_values(
        "importance",
        ascending=False
    )

    print()
    print("=" * 70)
    print(f"FEATURE IMPORTANCE: {feature_set_name}")
    print("=" * 70)

    print(
        importance.to_string(index=False)
    )


FEATURE IMPORTANCE: Semantic + Explicit
                      feature  importance
       job_description_length    0.078940
            resume_word_count    0.074521
          job_education_level    0.067724
                resume_length    0.064911
              job_skill_count    0.064509
               job_word_count    0.059963
          semantic_similarity    0.055714
               skill_coverage    0.054727
              education_match    0.052570
job_required_experience_years    0.047230
      resume_experience_years    0.046235
          resume_degree_count    0.044598
       resume_education_level    0.042668
         resume_core_cs_count    0.041971
           resume_skill_count    0.038725
          matched_skill_count    0.038255
            job_core_cs_count    0.036935
            skill_match_ratio    0.034947
             job_degree_count    0.029946
             experience_match    0.024910

FEATURE IMPORTANCE: All Features
                      feature  importance
 

In [43]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

print(importance.to_string(index=False))

                      feature  importance_mean  importance_std
       job_description_length         0.047129        0.004340
          semantic_similarity         0.015218        0.002953
            resume_word_count         0.009742        0.004326
               job_word_count         0.009178        0.002022
             tfidf_similarity         0.007882        0.005709
                resume_length         0.007802        0.004549
              job_skill_count         0.005788        0.003453
                 word_overlap         0.003632        0.004111
      resume_experience_years         0.001939        0.002805
            job_core_cs_count         0.001682        0.002361
            skill_match_ratio         0.001431        0.001345
          resume_degree_count         0.001331        0.001523
          matched_skill_count         0.000875        0.000339
       resume_education_level         0.000663        0.001021
             experience_match         0.000516        0

In [44]:
[name for name, value in globals().items()
 if hasattr(value, "columns")]

['_',
 '__',
 '___',
 'df',
 'X',
 '_4',
 'normalized_df',
 'X_train',
 'X_test',
 'results_df',
 '_20',
 '_21',
 'rf_results_df',
 '_24',
 'xgb_results_df',
 '_28',
 'gb_results_df',
 '_32',
 'tuned_results_df',
 'importance']

In [45]:
# ============================================================
# LABEL DISTRIBUTION BY SOURCE
# ============================================================

print(
    normalized_df.groupby(
        "source_dataset"
    )["label"]
    .value_counts(
        normalize=True
    )
    .unstack()
    .round(3)
)

label                   0      1
source_dataset                  
job_applicant       0.529  0.471
recruiter_decision  0.507  0.493


In [46]:
print(
    df.groupby("label")[
        "job_description_length"
    ].agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ]).round(2)
)

       count    mean  median     std  min   max
label                                          
0       5018  489.06   131.0  865.61   69  5088
1       4758  507.50   131.0  913.92   69  5269


In [47]:
suspicious_features = [
    "job_description_length",
    "resume_word_count",
    "job_word_count",
    "resume_length"
]

for feature in suspicious_features:

    print()
    print("=" * 60)
    print(feature)

    print(
        df.groupby("label")[feature]
        .agg(["mean", "median", "std"])
        .round(2)
    )


job_description_length
         mean  median     std
label                        
0      489.06   131.0  865.61
1      507.50   131.0  913.92

resume_word_count
         mean  median     std
label                        
0      286.40   367.0  171.85
1      287.95   368.0  164.50

job_word_count
        mean  median     std
label                       
0      69.72    21.0  121.15
1      72.35    21.0  127.48

resume_length
          mean  median      std
label                          
0      2126.42  2739.0  1258.49
1      2145.08  2754.0  1213.30


Its taking job description too seriously like if job description is big then accept.

In [48]:
print(
    normalized_df.groupby(
        "source_dataset"
    )["label"]
    .value_counts(
        normalize=True
    )
    .unstack()
    .round(3)
)

label                   0      1
source_dataset                  
job_applicant       0.529  0.471
recruiter_decision  0.507  0.493


In [49]:
print(
    normalized_df.groupby(
        "source_dataset"
    )["job_description"].apply(
        lambda x: x.str.len().mean()
    )
)

source_dataset
job_applicant         647.852973
recruiter_decision    439.660270
Name: job_description, dtype: float64


In [50]:
explicit_features = [
    "resume_skill_count",
    "job_skill_count",
    "matched_skill_count",
    "skill_match_ratio",
    "skill_coverage",
    "resume_education_level",
    "job_education_level",
    "education_match",
    "resume_experience_years",
    "job_required_experience_years",
    "experience_match",
    "resume_core_cs_count",
    "job_core_cs_count",
    "resume_degree_count",
    "job_degree_count",
]

print("=" * 70)
print("ZERO-RATE OF EXPLICIT FEATURES")
print("=" * 70)

for feature in explicit_features:

    zero_rate = (
        (df[feature] == 0)
        .mean()
    )

    print(
        f"{feature:35} "
        f"{zero_rate:.2%} zeros"
    )

ZERO-RATE OF EXPLICIT FEATURES
resume_skill_count                  12.03% zeros
job_skill_count                     58.25% zeros
matched_skill_count                 70.17% zeros
skill_match_ratio                   70.17% zeros
skill_coverage                      70.17% zeros
resume_education_level              26.59% zeros
job_education_level                 87.54% zeros
education_match                     90.49% zeros
resume_experience_years             54.93% zeros
job_required_experience_years       89.38% zeros
experience_match                    1.74% zeros
resume_core_cs_count                37.35% zeros
job_core_cs_count                   84.14% zeros
resume_degree_count                 28.28% zeros
job_degree_count                    87.54% zeros


In [51]:
# ============================================================
# INSPECT ZERO JOB-SKILL EXTRACTION
# ============================================================

zero_skill_indices = df.index[
    df["job_skill_count"] == 0
]

print(
    f"Rows with zero extracted job skills: "
    f"{len(zero_skill_indices)}"
)

for i in zero_skill_indices[:10]:

    row = normalized_df.iloc[i]
    features = df.iloc[i]

    print("=" * 80)
    print(f"ROW: {i}")
    print(f"SOURCE: {row['source_dataset']}")
    print(f"ROLE: {row['job_role']}")
    print(f"EXTRACTED SKILLS: {features['job_skill_count']}")
    print(
        f"EDUCATION: "
        f"{features['job_education_level']}"
    )
    print(
        f"EXPERIENCE: "
        f"{features['job_required_experience_years']}"
    )
    print()
    print(row["job_description"][:1000])

Rows with zero extracted job skills: 5695
ROW: 0
SOURCE: job_applicant
ROLE: Software Engineer
EXTRACTED SKILLS: 0.0
EDUCATION: 0.0
EXPERIENCE: 0.0

As a Software Engineer, you will leverage your advanced programming skills to develop cutting-edge software solutions that shape the future of technology. You will work on complex coding challenges, create robust systems, and contribute to innovative projects that require deep technical knowledge and analytical skills. This role requires a keen eye for logic and problem-solving, typically suited for individuals who enjoy working independently and thrive in high-tech environments. The role is perfect for someone with a strong interest in software development, system design, and engineering principles. Your work will directly impact the success of major technological products and services.
ROW: 2
SOURCE: job_applicant
ROLE: Software Engineer
EXTRACTED SKILLS: 0.0
EDUCATION: 0.0
EXPERIENCE: 0.0

As a Software Engineer, you will leverage your 

In [52]:
diagnostic_df = df.copy()

diagnostic_df["source_dataset"] = normalized_df[
    "source_dataset"
].values

print("Feature matrix rows:", len(df))
print("Normalized dataset rows:", len(normalized_df))
print("Diagnostic rows:", len(diagnostic_df))

Feature matrix rows: 9776
Normalized dataset rows: 9776
Diagnostic rows: 9776


In [53]:
# ============================================================
# ZERO-RATE BY SOURCE
# ============================================================

explicit_features = [
    "resume_skill_count",
    "job_skill_count",
    "matched_skill_count",
    "skill_match_ratio",
    "skill_coverage",
    "resume_education_level",
    "job_education_level",
    "education_match",
    "resume_experience_years",
    "job_required_experience_years",
    "experience_match",
    "resume_core_cs_count",
    "job_core_cs_count",
    "resume_degree_count",
    "job_degree_count"
]

zero_rates_by_source = (
    diagnostic_df.groupby("source_dataset")[explicit_features]
      .apply(lambda x: (x == 0).mean() * 100)
      .round(2)
)

print(zero_rates_by_source)

                    resume_skill_count  job_skill_count  matched_skill_count  \
source_dataset                                                                 
job_applicant                    37.43            35.61                 66.0   
recruiter_decision                2.13            67.08                 71.8   

                    skill_match_ratio  skill_coverage  resume_education_level  \
source_dataset                                                                  
job_applicant                    66.0            66.0                   61.00   
recruiter_decision               71.8            71.8                   13.18   

                    job_education_level  education_match  \
source_dataset                                             
job_applicant                     92.63            97.23   
recruiter_decision                85.56            87.86   

                    resume_experience_years  job_required_experience_years  \
source_dataset                     

In [54]:
# ============================================================
# ZERO-RATE BY LABEL
# ============================================================

zero_rate_by_label = (
    df.groupby("label")[explicit_features]
      .apply(lambda x: (x == 0).mean() * 100)
      .round(2)
)

print(zero_rate_by_label)

       resume_skill_count  job_skill_count  matched_skill_count  \
label                                                             
0                   11.80            58.21                69.89   
1                   12.27            58.30                70.47   

       skill_match_ratio  skill_coverage  resume_education_level  \
label                                                              
0                  69.89           69.89                   26.80   
1                  70.47           70.47                   26.36   

       job_education_level  education_match  resume_experience_years  \
label                                                                  
0                    88.10            90.75                    56.08   
1                    86.95            90.21                    53.72   

       job_required_experience_years  experience_match  resume_core_cs_count  \
label                                                                          
0        

In [55]:
# ============================================================
# EXPLICIT FEATURE MEANS BY LABEL
# ============================================================

print(
    df.groupby("label")[explicit_features]
      .mean()
      .T
      .round(3)
)

label                              0      1
resume_skill_count             6.141  6.268
job_skill_count                1.075  1.079
matched_skill_count            0.625  0.680
skill_match_ratio              0.244  0.253
skill_coverage                 0.129  0.120
resume_education_level         2.652  2.660
job_education_level            0.383  0.422
education_match                0.092  0.098
resume_experience_years        2.331  2.465
job_required_experience_years  0.520  0.594
experience_match               0.976  0.974
resume_core_cs_count           0.999  1.004
job_core_cs_count              0.205  0.178
resume_degree_count            1.030  1.029
job_degree_count               0.185  0.203


More analysis cuz our dataset sucks ig

In [56]:
# ============================================================
# LABEL REASON ANALYSIS
# ============================================================

print("=" * 70)
print("DECISION REASON BY LABEL")
print("=" * 70)

print(
    normalized_df.groupby("label")["decision_reason"]
    .value_counts()
    .head(20)
)

print()
print("=" * 70)
print("CLASSIFICATION REASON BY LABEL")
print("=" * 70)

print(
    normalized_df.groupby("label")["classification_reason"]
    .value_counts()
    .head(20)
)

DECISION REASON BY LABEL
label  decision_reason                                                         
0      No experience in back-end development.                                      497
       Needs improvement in machine learning algorithms.                           485
       Insufficient system design expertise for senior role.                       481
       Lacked leadership skills for a senior position.                             479
       Lacks hands-on experience with cloud platforms.                             440
       technical knowledge                                                          67
       cultural fit                                                                 67
       experience                                                                   59
       Lack of relevant skills or experience.                                       54
       Lack of enthusiasm or motivation.                                            48
       Unsatisfactory ref

In [57]:
# ============================================================
# LABEL DISTRIBUTION BY SOURCE
# ============================================================

print(
    pd.crosstab(
        normalized_df["source_dataset"],
        normalized_df["label"],
        normalize="index"
    ).round(3)
)

label                   0      1
source_dataset                  
job_applicant       0.529  0.471
recruiter_decision  0.507  0.493


In [58]:
# ============================================================
# COMPARE POSITIVE / NEGATIVE EXAMPLES
# ============================================================

for label in [0, 1]:

    print()
    print("=" * 80)
    print(f"LABEL = {label}")
    print("=" * 80)

    samples = normalized_df[
        normalized_df["label"] == label
    ].sample(5, random_state=42)

    for _, row in samples.iterrows():

        print("-" * 80)
        print("SOURCE:", row["source_dataset"])
        print("ROLE:", row["job_role"])

        print("\nRESUME:")
        print(row["candidate_resume"][:700])

        print("\nJOB:")
        print(row["job_description"][:700])

        print("\nDECISION REASON:")
        print(row["decision_reason"])

        print("\nCLASSIFICATION REASON:")
        print(row["classification_reason"])


LABEL = 0
--------------------------------------------------------------------------------
SOURCE: recruiter_decision
ROLE: AR/VR Developer

RESUME:
Here's a professional resume for Eric Jordan: Eric Jordan Contact Information: * Phone: (123) 456-7890 * Email: [eric.jordan@email.com](mailto:eric.jordan@email.com) * LinkedIn: linkedin.com/in/ericjordandev * GitHub: github.com/ericjordan Professional Summary: Highly motivated and detail-oriented AR/VR developer with 4+ years of experience in building immersive experiences using Unity, Oculus SDK, and Augmented Reality Markers. Skilled in creating interactive and engaging applications for various industries, including gaming, education, and entertainment. Proficient in 3D modeling, texturing, and animation. Technical Skills: * Programming languages: C#, UnityScript, JavaScript * Game Engine

JOB:
Be part of a passionate team at the forefront of machine learning as a AR/VR Developer, delivering solutions that shape the future.

DECISION R

In [60]:
print(normalized_df.columns.tolist())
print(normalized_df.head(2).T)

['candidate_resume', 'job_role', 'job_description', 'label', 'source_dataset', 'decision_reason', 'job_type', 'classification_reason', 'normalized_resume', 'normalized_job_description', 'normalized_job_role']
                                                                            0  \
candidate_resume            Proficient in Java, Algorithms, Problem Solvin...   
job_role                                                    Software Engineer   
job_description             As a Software Engineer, you will leverage your...   
label                                                                       1   
source_dataset                                                  job_applicant   
decision_reason                                                           NaN   
job_type                                                                 TECH   
classification_reason                       technical role: software engineer   
normalized_resume           proficient in java, algorithms, pr